In [1]:
import pandas as pd
import numpy as np
import os
import sys
%load_ext autoreload
%autoreload 2
# sys.path.append('/home/wolfgang/repos/sleep_research_io')
# from sleep_research_functions import *
%matplotlib widget
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu, kruskal, f_oneway
from icu_sleep_breathing_2020_help_functions import * 

pd.options.display.max_rows = 300
pd.options.display.max_columns = 300

font = {'family' : 'sans-serif',
        'weight' : 'normal',
        'size'   : 7}

font_headers =  {'family' : 'sans-serif',
        'weight' : 'normal',
        'size'   : 8}

matplotlib.rc('font', **font)

In [2]:
plots_savedir = '/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/plots'

[summary_subjects_icu, summary_subjects_sleeplab, 
 summary_days_icu, summary_days_sleeplab] = load_summary_data_with_inclusion_criteria()

/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3338: DtypeWarning: Columns (15,91,92,93,6455,6456,6457,12819,12820,12821,19185,19186,19187,21491,21492,21493,21519,21520,21521,21526,21527,21528,25549,25550,25551,31913,31914,31915,38279,38280,38281,44643,44644,44645,46851,46852,46853,46865,46866,46867,46879,46880,46881,46949,46950,46951,46956,46957,46958,46963,46964,46965,46970,46971,46972,46977,46978,46979,46984,46985,46986,51007,51008,51009,53336,53337,53338,57305,57373,57374,57375,59588,59589,59590,59602,59603,59604,59686,59687,59688,63669,63737,63738,63739,65952,65953,65954,65959,65960,65961,65980,65981,65982,66036,66037,66038,66043,66044,66045,66050,66051,66052,66057,66058,66059,66071,66072,66073,66078,66079,66080,70033,70101,70102,70103,72318,72319,72320,72332,72333,72334,72402,72403,72404,72416,72417,72418,76399,76467,76468,76469,78682,78683,78684,78696,78697,78698,78710,78711,78712,78766,78767,78768,78787,78788,78789,78801,7880

# of subjects ICU before inclusion criteria: 102
# of 12-hour segments ICU before inclusion criteria: 1161
# of subjects ICU after inclusion criteria: 102
# of 12-hour segments ICU after inclusion criteria: 1161
24-hour segments: 387
12-hour segments: 774

# of subjects sleeplab before inclusion criteria: 412
# of 12-hour segments sleeplab before inclusion criteria: 412


In [3]:
for agreement in ['all', 'agreement_relaxed', 'disagreement_relaxed', 'agreement', 'disagreement']:
    for modality in ['amendsumeffort', 'ecg_nn']:
        var_n2 = 'perc_N2_MODALITY_AGREEMENT'.replace('MODALITY', modality).replace('AGREEMENT', agreement)
        var_n3 = 'perc_N3_MODALITY_AGREEMENT'.replace('MODALITY', modality).replace('AGREEMENT', agreement)
        var_sum = 'perc_N2N3_MODALITY_AGREEMENT'.replace('MODALITY', modality).replace('AGREEMENT', agreement)

        summary_days_icu[var_sum] = summary_days_icu[var_n2] + summary_days_icu[var_n3]
        summary_days_sleeplab[var_sum] = summary_days_sleeplab[var_n2] + summary_days_sleeplab[var_n3]

In [4]:
for agreement in ['agreement_relaxed', 'agreement']:
    for modality in ['amendsumeffort', 'ecg_nn']:
        var_sleep = 'hours_sleep_MODALITY_all'.replace('MODALITY', modality) # all overlap sleep
        var_agreeing = 'hours_sleep_MODALITY_AGREEMENT'.replace('MODALITY', modality).replace('AGREEMENT', agreement)
        
        if agreement == 'agreement_relaxed':
             # all overlap sleep
            var_discordant = 'hours_discordant_MODALITY_AGREEMENT'.replace('MODALITY', modality).replace('AGREEMENT', agreement) # discordant sleep among overlap sleep
        elif agreement == 'agreement':
            var_discordant = 'hours_discordant_MODALITY_AGREEMENT'.replace('MODALITY', modality).replace('AGREEMENT', agreement)

            
        summary_days_icu[var_discordant] = summary_days_icu[var_sleep] - summary_days_icu[var_agreeing]
        summary_days_sleeplab[var_discordant] = summary_days_sleeplab[var_sleep] - summary_days_sleeplab[var_agreeing]
        
        
        sleep_total =  f'hours_sleep_{modality}_all'
        sleep_agree = f'hours_sleep_{modality}_{agreement}'
        sleep_discordant = f'hours_discordant_{modality}_{agreement}'
        discordant_proportion = f'discordant_proportion_{modality}_{agreement}'
        
        summary_days_icu[discordant_proportion] = np.divide(summary_days_icu[sleep_discordant], summary_days_icu[sleep_total])

In [5]:
summary_days_icu.loc[:, ['stages_distribution' in x for x in summary_days_icu.columns]] *= 100
summary_days_sleeplab.loc[:, ['stages_distribution' in x for x in summary_days_sleeplab.columns]] *= 100

summary_f_icu = summary_days_icu.loc[summary_days_icu.day_cat == 'f', :]
summary_dn_icu = summary_days_icu.loc[summary_days_icu.day_cat != 'f', :]

summary_days_sleeplab_full = summary_days_sleeplab.copy()

In [6]:
# compare ICU group 2 and sleeplab group 2:
# agreement = 'all'
# min_hours_sleep_icu = 0.01
# inclusion_agreement = 'all'
# do_agreement_for_sleeplab = False

# # ### compare ICU group 3 and sleeplab group 3, compute sleep statistics on ALL sleep:
agreement = 'all'
min_hours_sleep_icu = 2
inclusion_agreement = 'agreement_relaxed'
do_agreement_for_sleeplab = True

# ### compare ICU group 3 and sleeplab group 3, compute sleep statistics on CONCORDANT sleep:
# agreement = 'agreement_relaxed'
# min_hours_sleep_icu = 2
# inclusion_agreement = 'agreement_relaxed'
# do_agreement_for_sleeplab = True

In [7]:
days = 'full'

min_hours_sleep_icu = 2
if days == 'day_night':
    Xy_icu = summary_dn_icu.loc[np.any([summary_dn_icu[f'hours_sleep_amendsumeffort_{inclusion_agreement}'] >= min_hours_sleep_icu, 
                                                   summary_dn_icu[f'hours_sleep_ecg_nn_{inclusion_agreement}'] >= min_hours_sleep_icu],
                                                   axis=0), :].copy()
elif days == 'full':
    Xy_icu = summary_f_icu.loc[np.any([summary_f_icu[f'hours_sleep_amendsumeffort_{inclusion_agreement}'] >= min_hours_sleep_icu, 
                                                   summary_f_icu[f'hours_sleep_ecg_nn_{inclusion_agreement}'] >= min_hours_sleep_icu],
                                                   axis=0), :].copy()

print(f'Segments: {Xy_icu.shape[0]}')
print(f'Subjects: {Xy_icu.study_id.unique().shape[0]}')

Segments: 163
Subjects: 80


In [8]:
Xy_icu_subject = Xy_icu.loc[:, Xy_icu.dtypes != 'object'].copy()
for study_id_tmp in Xy_icu.study_id.unique():
    
    new_values = pd.DataFrame(Xy_icu_subject.loc[Xy_icu_subject.study_id == study_id_tmp].mean()).transpose()
    Xy_icu_subject = Xy_icu_subject[Xy_icu_subject.study_id != study_id_tmp]
    Xy_icu_subject = pd.concat([Xy_icu_subject, new_values], axis=0, ignore_index=True)

In [9]:
### 3x2 plot
# cols: data type, rows: stage version (resp, ecg, expert)

variables_template = [
 'hours_sleep_MODALITY_all', # overall sleep duration.
 'hours_sleep_MODALITY_agreement_relaxed',
 'hours_discordant_MODALITY_agreement_relaxed', 
 'discordant_proportion_MODALITY_agreement_relaxed',
 'perc_S_MODALITY_AGREEMENT',
 'perc_R_MODALITY_AGREEMENT',
 'perc_N1_MODALITY_AGREEMENT',
 'perc_N2_MODALITY_AGREEMENT',
 'perc_N3_MODALITY_AGREEMENT',
 'perc_N2N3_MODALITY_AGREEMENT',
 'sfi_MODALITY_AGREEMENT',
 'arousali_MODALITY_AGREEMENT',
]

x_labels = ['Total Sleep (h)', 'Concordant S. (h)', 'Discordant S. (h)', 'Discordant S. (%)', 'Sleep (%)', 'Stage R (%)', 'Stage N1 (%)', 'Stage N2 (%)', 'Stage N3 (%)', 'SFI', 'Wake Trans./h']
x_labels_short = ['TS', 'CS', 'DS', 'DS%' 'S', 'R', 'N1', 'N2', 'N3', 'SFI', 'WT']


### load the diagnosis categorization and add to Xy_icu

In [10]:
diagnosis_df = pd.read_excel('/scr/wolfgang/Dropbox (Partners HealthCare)/SleepInICUandSleeplabPaper/Analysis/summary_subjects_icu_inclusion_diagnoses_deid.xlsx', engine='openpyxl')
diagnosis2_df = pd.read_excel('/scr/wolfgang/Dropbox (Partners HealthCare)/SleepInICUandSleeplabPaper/Analysis/summary_subjects_icu_inclusion_diagnoses_deid.xlsx', sheet_name=3, header=None, engine='openpyxl')

print(diagnosis_df['sepsis'].unique())
diagnosis_df['sepsis'] = diagnosis_df['sepsis'] == 'yes'
print(diagnosis_df['respiratory_failure'].unique())
diagnosis_df['resp_failure'] = diagnosis_df['respiratory_failure'] == 'yes'

print(diagnosis_df['hemorrhage_or_shock'].unique())
diagnosis_df['hemorrhage'] = np.isin(diagnosis_df['hemorrhage_or_shock'].values, ['hemorrhage', 'shock and hemorrhage', 'hemorrhage and shock', 'hemorrhage, shock', 'yes'])
diagnosis_df['shock'] = np.isin(diagnosis_df['hemorrhage_or_shock'].values, ['shock', 'shock and hemorrhage', 'hemorrhage and shock', 'hemorrhage, shock', 'yes'])

diagnosis_df['sp_liver_transplant'] = diagnosis_df['s/p_liver_transplant'] == 'yes'

diagnosis_df['resp_disease'] = np.isin(diagnosis_df['resp_disease_norespfailure'].values,['yes', 'yes?'])

diagnosis_df['heart_failure'] = np.isin(diagnosis_df['heart_failure'].values,['yes'])

diagnoses = ['resp_failure', 'resp_disease', 'heart_failure', 'sepsis', 'hemorrhage', 'shock', 'sp_liver_transplant']

diagnosis_df['other'] = ~ diagnosis_df[diagnoses].any(axis=1)

diagnoses += ['other'] 

[nan 'yes']
[nan 'yes']
['shock' nan 'yes' 'shock and hemorrhage' 'hemorrhage'
 'hemorrhage and shock' 'hemorrhage, shock']


# new tables/sheets from parimala

In [11]:
idx_medical = np.where(diagnosis2_df[0] == 'Medical highlighted conditions')[0][0]
idx_surgical = np.where(diagnosis2_df[0] == 'Surgical')[0][0]
idx_unhighlighted = np.where(diagnosis2_df[0] == 'Unhighlighted conditions')[0][0]

In [12]:
diagnosis_medical = diagnosis2_df.iloc[idx_medical : idx_surgical, :].transpose()
diagnosis_medical.columns = diagnosis_medical.iloc[0]
diagnosis_medical.drop(0, axis=0, inplace=True)
diagnosis_medical.drop(['Medical highlighted conditions'], axis=1, inplace=True)

diagnosis_surgical = diagnosis2_df.iloc[idx_surgical : idx_unhighlighted, :].transpose()
diagnosis_unhighlighted = diagnosis2_df.iloc[idx_unhighlighted, :].transpose()

diagnosis_surgical.columns = diagnosis_surgical.iloc[0]
diagnosis_surgical.drop(0, axis=0, inplace=True)
diagnosis_surgical.drop(['Surgical'], axis=1, inplace=True)

diagnosis_unhighlighted = diagnosis2_df.iloc[idx_unhighlighted, :].transpose()

In [13]:
diagnosis_medical.head()

,GI hemorrhage,Shock,Sepsis,Heart failure,Anemia,Hepatic encephalopathy,Hemorrhage,Encephalopathy,Pneumonia,UTI,Resp failure,AKI,AMS,COPD exacerbation,NSTEMI,Pneumothorax,Mixed resp & met acidosis,ILD,Pulm edema,Pl eff,SBO,ARDS,Panenterocolitis,Fall,SAH,MVC,Heart block,Gastric outlet obstruction,Hemothorax,SIRS,ATN,Trauma,Resp insufficiency,IPH,Cardiorenal synd,GB perf,Cryptogenic hep cirrhosis,Pulm emboli,HTN emergency,PRES,Parkinson's disease SIADH,Suicidal ideation,Parox VT,Subdural hematoma,PEA,Acute epiglottitis,Flash facial burn injury,Aortic dissection,NaN,NaN
1,1.0,1.0,3.0,3.0,12.0,21.0,24.0,25.0,25.0,3.0,25.0,3.0,33.0,45.0,45.0,49.0,69.0,79.0,83.0,84.0,86.0,88.0,109.0,121.0,121.0,122.0,122.0,126.0,128.0,131.0,133.0,135.0,135.0,121.0,139.0,140.0,145.0,151.0,155.0,155.0,156.0,157.0,158.0,165.0,176.0,177.0,178.0,186.0,NaN,NaN
2,14.0,3.0,12.0,47.0,14.0,86.0,123.0,77.0,33.0,12.0,31.0,15.0,56.0,47.0,82.0,52.0,NaN,111.0,176.0,145.0,NaN,178.0,NaN,137.0,137.0,128.0,NaN,NaN,179.0,NaN,NaN,NaN,160.0,NaN,NaN,NaN,NaN,154.0,161.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,82.0,12.0,25.0,49.0,18.0,180.0,133.0,NaN,45.0,25.0,33.0,31.0,77.0,91.0,180.0,NaN,NaN,158.0,178.0,179.0,NaN,180.0,NaN,165.0,NaN,NaN,NaN,NaN,187.0,NaN,NaN,NaN,169.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,83.0,15.0,31.0,79.0,21.0,NaN,135.0,NaN,46.0,63.0,45.0,33.0,79.0,NaN,NaN,NaN,NaN,164.0,181.0,187.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,24.0,33.0,82.0,25.0,NaN,149.0,NaN,49.0,101.0,46.0,34.0,101.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
diagnosis_surgical.head()

,Cervical stenosis with complete quadriplegia,Jugular foramen schwannoma,Nephrolithiasis,Myaesthenia gravis,Bronchopleural fistula,Incarcerated umb hernia,PAD,Aortic dissection,Necrotising soft tissue inf,Retroperitoneal hemorrhage,AAA,Cord compression,Gastric perf,Craniopharyngioma,Vocal fold necrosis & pharyngeal stenosis,MVA with multiple rib fractures,Perf sigmoid colon,Lung adenoca with LUL lobectomy,Degenerative lumbar stenosis,Lung tumor thoracotomy with chest wall resection,SBO,Bladder rupture,Ischemic stroke,Eso perf,HCV/alc cirrhosis s/p liver transplant,Carotid art sten,HCV cirrhosis HCC s/p liver transplant,Spinal myxoid chondrosarcoma,HCV cirrhosis cholangioca HCC s/p liver transpl,Bilateral hydronephrosis,Hip fracture,NASH cirrhosis s/p liver transplant,Ischemic colitis,Scoliosis,Cervical spondylotic myelopathy,ESRD PKD s/p renal transplant,Meningioma,Bladder carcinoma,Pancreatic cancer,HCC s/p IR hep art embolization,Sacral chordoma,Sigmoid diverticulitis,Fall,Degenerative lumbar kyphosis,NaN,NaN
1,6.0,11.0,12.0,13.0,18.0,21.0,24.0,29.0,34.0,35.0,37.0,48.0,49.0,50.0,51.0,52.0,56.0,63.0,68.0,69.0,72.0,74.0,81.0,84.0,87.0,90.0,97.0,100.0,103.0,130.0,131.0,133.0,139.0,68.0,153.0,160.0,167.0,168.0,169.0,171.0,173.0,179.0,187.0,188.0,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,175.0,NaN,142.0,NaN,54.0,157.0,NaN,NaN,NaN,NaN,182.0,NaN,143.0,NaN,83.0,NaN,NaN,NaN,NaN,161.0,104.0,NaN,NaN,NaN,NaN,NaN,180.0,188.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,189.0,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,183.0,NaN,NaN,NaN,89.0,176.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,149.0,NaN,NaN,NaN,NaN,NaN,120.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
study_ids_medical = Xy_icu_subject.loc[Xy_icu_subject.med_surg == 1, 'study_id'].values
study_ids_surgical = Xy_icu_subject.loc[Xy_icu_subject.med_surg == 0, 'study_id'].values

In [16]:
diagnoses = {}
diagnoses['sepsis'] = list(diagnosis_medical['Sepsis'].dropna())
diagnoses['shock'] = list(diagnosis_medical['Shock'].dropna())
diagnoses['heart_failure'] = list(diagnosis_medical['Heart failure'].dropna())
diagnoses['encephalopathy'] = list(diagnosis_medical['Hepatic encephalopathy'].dropna()) + list(diagnosis_medical['Encephalopathy'].dropna()) + list(diagnosis_medical['AMS'].dropna())
diagnoses['anemia'] = list(diagnosis_medical['Anemia'].dropna())
diagnoses['hemorrhage'] = list(diagnosis_medical['Hemorrhage'].dropna()) + list(diagnosis_medical['GI hemorrhage'].dropna())
diagnoses['pneumonia'] = list(diagnosis_medical['Pneumonia'].dropna())
diagnoses['respiratory_failure'] = list(diagnosis_medical['Resp failure'].dropna())
diagnoses['aki'] = list(diagnosis_medical['AKI'].dropna())
diagnoses['pneumothorax'] = list(diagnosis_medical['Pneumothorax'].dropna()) + list(diagnosis_medical['Pulm edema'].dropna()) + list(diagnosis_medical['Hemothorax'].dropna()) + list(diagnosis_medical['Pl eff'].dropna())
diagnoses['copd'] = list(diagnosis_medical['COPD exacerbation'].dropna()) + list(diagnosis_medical['ILD'].dropna())
other_medical = set(study_ids_medical)
for diagnosis in diagnoses.keys():
    other_medical = other_medical - set(diagnoses[diagnosis])
diagnoses['other_medical'] = list(other_medical)

diagnoses['gi_hernia_ischemic'] = list(diagnosis_surgical['Gastric perf'].dropna()) + list(diagnosis_surgical['Incarcerated umb hernia'].dropna()) + list(diagnosis_surgical['Perf sigmoid colon'].dropna()) + list(diagnosis_surgical['Eso perf'].dropna()) + list(diagnosis_surgical['Sigmoid diverticulitis'].dropna()) + list(diagnosis_surgical['Ischemic colitis'].dropna())
diagnoses['cirrhosis_liver'] = list(diagnosis_surgical['HCV cirrhosis cholangioca HCC s/p liver transpl'].dropna()) + list(diagnosis_surgical['HCV cirrhosis HCC s/p liver transplant'].dropna()) + list(diagnosis_surgical['NASH cirrhosis s/p liver transplant'].dropna()) + list(diagnosis_surgical['Eso perf'].dropna()) + list(diagnosis_surgical['HCV/alc cirrhosis s/p liver transplant'].dropna())
other_surgical = set(study_ids_surgical)
for diagnosis in ['gi_hernia_ischemic', 'cirrhosis_liver']:
    other_surgical = other_surgical - set(diagnoses[diagnosis])
diagnoses['other_surgical'] = list(other_surgical)

In [17]:
Xy_icu_subject.iloc[8:14]

,inclusion_subject,study_id,mrn,age,bmi,sex,osa_diagnosis,icu_mortality,3_month_mortality,icu_los,hospital_los,chf,copd,med_surg,CCI,CCI_weighted,Belt_Length,day_no,hr_mean,hr_std,hr_median,hr_q25,hr_q75,hr_sda,hr_asd,spo2_mean,spo2_std,spo2_median,spo2_q25,spo2_q75,spo2_sda,spo2_asd,spo2_airgo_mean,spo2_airgo_std,spo2_airgo_median,spo2_airgo_q25,spo2_airgo_q75,spo2_airgo_sda,spo2_airgo_asd,spo2_clean_mean,spo2_clean_std,spo2_clean_median,spo2_clean_q25,spo2_clean_q75,spo2_clean_sda,spo2_clean_asd,spo2_airgo_clean_mean,spo2_airgo_clean_std,spo2_airgo_clean_median,spo2_airgo_clean_q25,spo2_airgo_clean_q75,spo2_airgo_clean_sda,spo2_airgo_clean_asd,edw_bp_systolic_mean,edw_bp_systolic_std,edw_bp_systolic_median,edw_bp_systolic_q25,edw_bp_systolic_q75,edw_bp_diastolic_mean,edw_bp_diastolic_std,edw_bp_diastolic_median,edw_bp_diastolic_q25,edw_bp_diastolic_q75,edw_bp_map_mean,edw_bp_map_std,edw_bp_map_median,edw_bp_map_q25,edw_bp_map_q75,cpc_hfc_lfc_ratio_mean,cpc_hfc_lfc_ratio_std,cpc_hfc_lfc_ratio_median,cpc_hfc_lfc_ratio_q25,cpc_hfc_lfc_ratio_q75,cpc_hfc_mean,cpc_hfc_std,cpc_hfc_median,cpc_hfc_q25,cpc_hfc_q75,cpc_hfc_sum,cpc_lfc_mean,cpc_lfc_std,cpc_lfc_median,cpc_lfc_q25,cpc_lfc_q75,cpc_lfc_sum,hrv_nn_q25,hrv_nn_q75,hrv_nn_sda,hrv_nn_asd,hco3_arterial_mean,hco3_arterial_min,hco3_arterial_max,pco2_arterial_mean,pco2_arterial_min,pco2_arterial_max,po2_arterial_mean,po2_arterial_min,po2_arterial_max,ph_arterial_mean,ph_arterial_min,ph_arterial_max,oxygen_flow_rate_max,sofa_score_mean,sofa_score_min,sofa_score_max,ahi_va_a3,ahi_va_a3_ss,ahi_vb_a3,ahi_vb_a3_ss,ahi_ro_a3,ahi_ro_a3_ss,ahi_rsr_a3,ahi_rsr_a3_ss,ahi_expert,ahi_va_a3_ss_aswti,ahi_vb_a3_ss_aswti,ahi_ro_a3_ss_aswti,ahi_rsr_a3_ss_aswti,hypoxic_burden_va_a3,hypoxic_burden_va_a3_ss,hypoxic_burden_vb_a3,hypoxic_burden_vb_a3_ss,hypoxic_burden_ro_a3,hypoxic_burden_ro_a3_ss,hypoxic_burden_rsr_a3,hypoxic_burden_rsr_a3_ss,hypoxic_burden_expert,hypoxic_burden_va_a3_ss_aswti,hypoxic_burden_vb_a3_ss_aswti,hypoxic_burden_ro_a3_ss_aswti,hypoxic_burden_rsr_a3_ss_aswti,hypoxia_LDI,hypoxia_SDI,hypoxia_TDI,hypoxia_pu90,hypoxia_ss_LDI,hypoxia_ss_SDI,hypoxia_ss_TDI,hypoxia_ss_pu90,hypoxia_ss_aswti_LDI,hypoxia_ss_aswti_SDI,hypoxia_ss_aswti_TDI,hypoxia_ss_aswti_pu90,self_similarity_sleep,self_similarity_sleep_q50,self_similarity_sleep_q75,self_similarity_apnea,self_similarity_apnea_q50,self_similarity_apnea_q75,opioids_sum,...,hours_data_ecg_nn_agreement_relaxed,hours_sleep_ecg_nn_agreement_relaxed,perc_W_ecg_nn_agreement_relaxed,perc_S_ecg_nn_agreement_relaxed,perc_R_ecg_nn_agreement_relaxed,perc_N1_ecg_nn_agreement_relaxed,perc_N2_ecg_nn_agreement_relaxed,perc_N3_ecg_nn_agreement_relaxed,sfi_ecg_nn_agreement_relaxed,sfi_w_ecg_nn_agreement_relaxed,arousali_ecg_nn_agreement_relaxed,hours_data_ecg_nn_disagreement_relaxed,hours_sleep_ecg_nn_disagreement_relaxed,perc_W_ecg_nn_disagreement_relaxed,perc_S_ecg_nn_disagreement_relaxed,perc_R_ecg_nn_disagreement_relaxed,perc_N1_ecg_nn_disagreement_relaxed,perc_N2_ecg_nn_disagreement_relaxed,perc_N3_ecg_nn_disagreement_relaxed,sfi_ecg_nn_disagreement_relaxed,sfi_w_ecg_nn_disagreement_relaxed,arousali_ecg_nn_disagreement_relaxed,hours_data_amendsumeffort_all,hours_sleep_amendsumeffort_all,perc_W_amendsumeffort_all,perc_S_amendsumeffort_all,perc_R_amendsumeffort_all,perc_N1_amendsumeffort_all,perc_N2_amendsumeffort_all,perc_N3_amendsumeffort_all,sfi_amendsumeffort_all,sfi_w_amendsumeffort_all,arousali_amendsumeffort_all,hours_data_amendsumeffort_agreement,hours_sleep_amendsumeffort_agreement,perc_W_amendsumeffort_agreement,perc_S_amendsumeffort_agreement,perc_R_amendsumeffort_agreement,perc_N1_amendsumeffort_agreement,perc_N2_amendsumeffort_agreement,perc_N3_amendsumeffort_agreement,sfi_amendsumeffort_agreement,sfi_w_amendsumeffort_agreement,arousali_amendsumeffort_agreement,hours_data_amendsumeffort_disagreement,hours_sleep_amendsumeffort_disagreement,perc_W_amendsumeffort_disagreement,perc_S_amendsumeffort_disagreement,perc_R_amendsumeffort_disagreement,p

In [18]:
Xy_icu_subject.shape

(80, 6521)

In [19]:
for diagnosis in diagnoses.keys():
    print(f"{diagnosis:>20}: {len(diagnoses[diagnosis])}")
    
    Xy_icu[diagnosis] = 0
    Xy_icu_subject[diagnosis] = 0
    
    for study_id_diagnosis_tmp in diagnoses[diagnosis]:
        Xy_icu.loc[Xy_icu.study_id == study_id_diagnosis_tmp, diagnosis] = 1
        Xy_icu_subject.loc[Xy_icu_subject.study_id == study_id_diagnosis_tmp, diagnosis] = 1

              sepsis: 19
               shock: 30
       heart_failure: 10
      encephalopathy: 14
              anemia: 20
          hemorrhage: 10
           pneumonia: 15
 respiratory_failure: 21
                 aki: 33
        pneumothorax: 13
                copd: 7
       other_medical: 3
  gi_hernia_ischemic: 8
     cirrhosis_liver: 7
      other_surgical: 43


In [20]:
variables_template

['hours_sleep_MODALITY_all',
 'hours_sleep_MODALITY_agreement_relaxed',
 'hours_discordant_MODALITY_agreement_relaxed',
 'discordant_proportion_MODALITY_agreement_relaxed',
 'perc_S_MODALITY_AGREEMENT',
 'perc_R_MODALITY_AGREEMENT',
 'perc_N1_MODALITY_AGREEMENT',
 'perc_N2_MODALITY_AGREEMENT',
 'perc_N3_MODALITY_AGREEMENT',
 'perc_N2N3_MODALITY_AGREEMENT',
 'sfi_MODALITY_AGREEMENT',
 'arousali_MODALITY_AGREEMENT']

## start analysis

In [21]:
if 0: # V1 of parimala table (sheet 1)

    print(f"{diagnosis_tmp}: {diagnosis_df[diagnosis_tmp].sum()}")
    Xy_icu = Xy_icu.merge(diagnosis_df[diagnoses + ['study_id']])
    Xy_icu_subject = Xy_icu_subject.merge(diagnosis_df[diagnoses + ['study_id']])
    


modality = 'amendsumeffort'
modality = 'ecg_nn'

discordant_proportion = f'discordant_proportion_{modality}_agreement_relaxed'
print(discordant_proportion)

variables = [x.replace('MODALITY', modality).replace('AGREEMENT', agreement) for x in variables_template]
print(variables)

sleep_vars = variables

discordant_proportion_ecg_nn_agreement_relaxed
['hours_sleep_ecg_nn_all', 'hours_sleep_ecg_nn_agreement_relaxed', 'hours_discordant_ecg_nn_agreement_relaxed', 'discordant_proportion_ecg_nn_agreement_relaxed', 'perc_S_ecg_nn_all', 'perc_R_ecg_nn_all', 'perc_N1_ecg_nn_all', 'perc_N2_ecg_nn_all', 'perc_N3_ecg_nn_all', 'perc_N2N3_ecg_nn_all', 'sfi_ecg_nn_all', 'arousali_ecg_nn_all']


In [22]:
variables

['hours_sleep_ecg_nn_all',
 'hours_sleep_ecg_nn_agreement_relaxed',
 'hours_discordant_ecg_nn_agreement_relaxed',
 'discordant_proportion_ecg_nn_agreement_relaxed',
 'perc_S_ecg_nn_all',
 'perc_R_ecg_nn_all',
 'perc_N1_ecg_nn_all',
 'perc_N2_ecg_nn_all',
 'perc_N3_ecg_nn_all',
 'perc_N2N3_ecg_nn_all',
 'sfi_ecg_nn_all',
 'arousali_ecg_nn_all']

In [23]:
df_results = pd.DataFrame(index = sleep_vars)

values_groups = {}
for sleep_var in sleep_vars:
    values_groups[sleep_var] = []

dec = [1]*3 + [2]*7 + [1]*2

for diagnosis in diagnoses:

    df_results[diagnosis + '_median'] = np.nan
    df_results[diagnosis + '_iqr'] = np.nan

    Xy_icu_subject_group = Xy_icu_subject.query(f"{diagnosis} == 1")
    
    df_results.loc['N', diagnosis + '_median'] = Xy_icu_subject_group.shape[0]
    
    for i, sleep_var in enumerate(sleep_vars):
        df_results.loc[sleep_var, f"{diagnosis}_median"] = str(np.round(Xy_icu_subject_group[sleep_var].median(), dec[i]))
        df_results.loc[sleep_var, f"{diagnosis}_iqr"] = f"[{Xy_icu_subject_group[sleep_var].quantile(0.25).round(dec[i])}, {Xy_icu_subject_group[sleep_var].quantile(0.75).round(dec[i])}]"

    for i, sleep_var in enumerate(sleep_vars):

        values_0 = Xy_icu_subject.query(f"{diagnosis} == 0")[sleep_var].values
        values_1 = Xy_icu_subject.query(f"{diagnosis} == 1")[sleep_var].values

        if 0:
            df_results.loc[sleep_var, f"{diagnosis}_mwu"] = f"{mannwhitneyu(values_0, values_1)[0].round(1), mannwhitneyu(values_0, values_1)[1].round(4)}"

            if mannwhitneyu(values_0, values_1)[1].round(4) < 0.01:
                print('** ' + sleep_var + ' ' + diagnosis)
            elif mannwhitneyu(values_0, values_1)[1].round(4) < 0.05:
                print('* ' + sleep_var + ' ' + diagnosis)

        values_groups[sleep_var].append(values_1)
        
        
for sleep_var in sleep_vars:
    values_tmp = values_groups[sleep_var]
    assert len(values_tmp) == len(diagnoses), 'For each diagnosis group, there must be one list of values'
    # apply kruskal test:
    df_results.loc[sleep_var, f"kruskal"] = f"{np.round(kruskal(*values_tmp)[0], 1)}, {np.round(kruskal(*values_tmp)[1], 3)}"

    


In [24]:
df_results

,sepsis_median,sepsis_iqr,shock_median,shock_iqr,heart_failure_median,heart_failure_iqr,encephalopathy_median,encephalopathy_iqr,anemia_median,anemia_iqr,hemorrhage_median,hemorrhage_iqr,pneumonia_median,pneumonia_iqr,respiratory_failure_median,respiratory_failure_iqr,aki_median,aki_iqr,pneumothorax_median,pneumothorax_iqr,copd_median,copd_iqr,other_medical_median,other_medical_iqr,gi_hernia_ischemic_median,gi_hernia_ischemic_iqr,cirrhosis_liver_median,cirrhosis_liver_iqr,other_surgical_median,other_surgical_iqr,kruskal
hours_sleep_ecg_nn_all,6.2,"[4.4, 7.5]",6.0,"[3.2, 7.5]",6.7,"[1.3, 8.3]",7.4,"[5.0, 7.6]",5.9,"[4.4, 7.3]",7.3,"[4.7, 9.5]",6.1,"[4.9, 7.5]",5.5,"[3.6, 9.0]",6.6,"[4.6, 8.7]",6.9,"[6.0, 7.9]",7.8,"[5.5, 10.0]",6.4,"[6.4, 10.7]",7.0,"[5.0, 7.5]",4.8,"[3.9, 6.2]",6.8,"[4.9, 9.2]","7.7, 0.904"
hours_sleep_ecg_nn_agreement_relaxed,3.5,"[2.1, 4.1]",3.7,"[1.8, 4.7]",3.2,"[1.1, 5.0]",3.7,"[2.7, 4.1]",3.3,"[2.7, 4.5]",4.2,"[3.2, 6.4]",2.7,"[1.5, 3.8]",2.6,"[1.5, 4.3]",3.7,"[2.1, 5.2]",3.5,"[2.7, 4.4]",4.9,"[3.0, 6.4]",3.4,"[2.5, 3.9]",3.2,"[2.8, 3.7]",2.7,"[2.6, 3.3]",4.1,"[2.8, 5.7]","12.5, 0.565"
hours_discordant_ecg_nn_agreement_relaxed,2.8,"[1.0, 3.8]",1.7,"[0.6, 3.8]",2.0,"[0.3, 3.8]",2.3,"[0.6, 4.7]",2.3,"[1.3, 3.1]",3.2,"[0.5, 4.1]",3.1,"[2.3, 4.5]",2.3,"[0.8, 5.3]",2.8,"[0.6, 3.8]",3.6,"[1.5, 5.3]",2.6,"[1.3, 3.5]",4.8,"[3.4, 8.3]",3.7,"[2.6, 4.2]",1.7,"[1.2, 3.5]",2.3,"[1.1, 3.7]","8.0, 0.888"
discordant_proportion_ecg_nn_agreement_relaxed,0.38,"[0.17, 0.5]",0.31,"[0.14, 0.45]",0.43,"[0.08, 0.61]",0.54,"[0.17, 0.62]",0.36,"[0.2, 0.46]",0.32,"[0.12, 0.42]",0.49,"[0.28, 0.59]",0.51,"[0.25, 0.59]",0.35,"[0.12, 0.54]",0.53,"[0.41, 0.63]",0.36,"[0.35, 0.45]",0.75,"[0.53, 0.75]",0.5,"[0.45, 0.55]",0.54,"[0.31, 0.54]",0.34,"[0.18, 0.48]","16.6, 0.278"
perc_S_ecg_nn_all,0.42,"[0.22, 0.53]",0.37,"[0.25, 0.49]",0.41,"[0.07, 0.52]",0.45,"[0.25, 0.53]",0.32,"[0.27, 0.51]",0.42,"[0.39, 0.69]",0.37,"[0.25, 0.53]",0.37,"[0.25, 0.54]",0.43,"[0.32, 0.64]",0.42,"[0.38, 0.53]",0.49,"[0.32, 0.57]",0.62,"[0.54, 0.71]",0.39,"[0.34, 0.46]",0.32,"[0.26, 0.32]",0.5,"[0.3, 0.66]","10.8, 0.705"
perc_R_ecg_nn_all,0.04,"[0.0, 0.13]",0.04,"[0.0, 0.2]",0.04,"[0.01, 0.08]",0.01,"[0.0, 0.13]",0.0,"[0.0, 0.06]",0.34,"[0.04, 0.7]",0.07,"[0.0, 0.26]",0.1,"[0.0, 0.43]",0.06,"[0.0, 0.22]",0.19,"[0.08, 0.25]",0.05,"[0.02, 0.09]",0.04,"[0.02, 0.31]",0.11,"[0.03, 0.23]",0.34,"[0.3, 0.63]",0.12,"[0.02, 0.34]","18.4, 0.191"
perc_N1_ecg_nn_all,0.33,"[0.16, 0.82]",0.28,"[0.07, 0.77]",0.26,"[0.04, 0.36]",0.33,"[0.17, 0.82]",0.31,"[0.12, 0.78]",0.36,"[0.18, 0.6]",0.31,"[0.24, 0.81]",0.18,"[0.04, 0.31]",0.22,"[0.08, 0.46]",0.15,"[0.08, 0.28]",0.09,"[0.04, 0.15]",0.09,"[0.06, 0.19]",0.17,"[0.13, 0.25]",0.16,"[0.15, 0.19]",0.17,"[0.06, 0.41]","16.6, 0.279"
perc_N2_ecg_nn_all,0.01,"[0.0, 0.06]",0.05,"[0.0, 0.13]",0.01,"[0.0, 0.04]",0.04,"[0.02, 0.07]",0.03,"[0.01, 0.13]",0.03,"[0.0, 0.1]",0.06,"[0.01, 0.09]",0.02,"[0.0, 0.08]",0.02,"[0.0, 0.08]",0.03,"[0.01, 0.12]",0.03,"[0.01, 0.14]",0.01,"[0.01, 0.02]",0.05,"[0.02, 0.06]",0.14,"[0.01, 0.33]",0.11,"[0.01, 0.21]","15.7, 0.33"
perc_N3_ecg_nn_all,0.41,"[0.06, 0.6]",0.3,"[0.05, 0.51]",0.59,"[0.32, 0.89]",0.17,"[0.05, 0.48]",0.37,"[0.06, 0.72]",0.06,"[0.01, 0.18]",0.17,"[0.06, 0.41]",0.23,"[0.07, 0.58]",0.44,"[0.1, 0.59]",0.57,"[0.42, 0.62]",0.86,"[0.64, 0.91]",0.91,"[0.51, 0.91]",0.6,"[0.42, 0.68]",0.09,"[0.08, 0.12]",0.24,"[0.05, 0.55]","21.9, 0.081"
perc_N2N3_ecg_nn_all,0.46,"[0.15, 0.72]",0.3,"[0.07, 0.75]",0.63,"[0.36, 0.91]",0.18,"[0.08, 0.7]",0.54,"[0.15, 0.84]",0.1,"[0.05, 0.39]",0.3,"[0.15, 0.52]",0.31,"[0.18, 0.6]",0.5,"[0.11, 0.73]",0.62,"[0.49, 0.7]",0.87,"[0.77, 0.94]",0.91,"[0.52, 0.92]",0.7,"[0.54, 0.74]",0.43,"[0.22, 0.5]",0.45,"[0.18, 0.77]","18.4, 0.189"


In [25]:
df_results.loc['perc_N2N3_ecg_nn_all', [x for x in df_results.columns if 'median' in x]].astype(float).sort_values()

hemorrhage_median             0.10
encephalopathy_median         0.18
shock_median                  0.30
pneumonia_median              0.30
respiratory_failure_median    0.31
cirrhosis_liver_median        0.43
other_surgical_median         0.45
sepsis_median                 0.46
aki_median                    0.50
anemia_median                 0.54
pneumothorax_median           0.62
heart_failure_median          0.63
gi_hernia_ischemic_median     0.70
copd_median                   0.87
other_medical_median          0.91
Name: perc_N2N3_ecg_nn_all, dtype: float64

In [26]:
df_results.loc['perc_N1_ecg_nn_all', [x for x in df_results.columns if 'median' in x]].astype(float).sort_values()

copd_median                   0.09
other_medical_median          0.09
pneumothorax_median           0.15
cirrhosis_liver_median        0.16
gi_hernia_ischemic_median     0.17
other_surgical_median         0.17
respiratory_failure_median    0.18
aki_median                    0.22
heart_failure_median          0.26
shock_median                  0.28
anemia_median                 0.31
pneumonia_median              0.31
sepsis_median                 0.33
encephalopathy_median         0.33
hemorrhage_median             0.36
Name: perc_N1_ecg_nn_all, dtype: float64

In [27]:
df_results.loc['perc_R_ecg_nn_all', [x for x in df_results.columns if 'median' in x]].astype(float).sort_values()

anemia_median                 0.00
encephalopathy_median         0.01
sepsis_median                 0.04
shock_median                  0.04
heart_failure_median          0.04
other_medical_median          0.04
copd_median                   0.05
aki_median                    0.06
pneumonia_median              0.07
respiratory_failure_median    0.10
gi_hernia_ischemic_median     0.11
other_surgical_median         0.12
pneumothorax_median           0.19
hemorrhage_median             0.34
cirrhosis_liver_median        0.34
Name: perc_R_ecg_nn_all, dtype: float64

In [28]:
df_results.loc['discordant_proportion_ecg_nn_agreement_relaxed', [x for x in df_results.columns if 'median' in x]].astype(float).sort_values()

shock_median                  0.31
hemorrhage_median             0.32
other_surgical_median         0.34
aki_median                    0.35
anemia_median                 0.36
copd_median                   0.36
sepsis_median                 0.38
heart_failure_median          0.43
pneumonia_median              0.49
gi_hernia_ischemic_median     0.50
respiratory_failure_median    0.51
pneumothorax_median           0.53
encephalopathy_median         0.54
cirrhosis_liver_median        0.54
other_medical_median          0.75
Name: discordant_proportion_ecg_nn_agreement_relaxed, dtype: float64

In [29]:
df_results.to_csv('sleep_statistics_diagnoses_groups.csv')
df_results.transpose().to_csv('sleep_statistics_diagnoses_groupsT.csv')

In [30]:
print(discordant_proportion)
df_results.loc[discordant_proportion]

discordant_proportion_ecg_nn_agreement_relaxed


sepsis_median                         0.38
sepsis_iqr                     [0.17, 0.5]
shock_median                          0.31
shock_iqr                     [0.14, 0.45]
heart_failure_median                  0.43
heart_failure_iqr             [0.08, 0.61]
encephalopathy_median                 0.54
encephalopathy_iqr            [0.17, 0.62]
anemia_median                         0.36
anemia_iqr                     [0.2, 0.46]
hemorrhage_median                     0.32
hemorrhage_iqr                [0.12, 0.42]
pneumonia_median                      0.49
pneumonia_iqr                 [0.28, 0.59]
respiratory_failure_median            0.51
respiratory_failure_iqr       [0.25, 0.59]
aki_median                            0.35
aki_iqr                       [0.12, 0.54]
pneumothorax_median                   0.53
pneumothorax_iqr              [0.41, 0.63]
copd_median                           0.36
copd_iqr                      [0.35, 0.45]
other_medical_median                  0.75
other_medic

In [31]:
# df_results = pd.read_csv('/scr/wolfgang/Dropbox (Partners HealthCare)/SleepInICUandSleeplabPaper/Results/sleep_statistics_diagnoses_groups_A2.csv', index_col=0)

In [32]:
df_results

,sepsis_median,sepsis_iqr,shock_median,shock_iqr,heart_failure_median,heart_failure_iqr,encephalopathy_median,encephalopathy_iqr,anemia_median,anemia_iqr,hemorrhage_median,hemorrhage_iqr,pneumonia_median,pneumonia_iqr,respiratory_failure_median,respiratory_failure_iqr,aki_median,aki_iqr,pneumothorax_median,pneumothorax_iqr,copd_median,copd_iqr,other_medical_median,other_medical_iqr,gi_hernia_ischemic_median,gi_hernia_ischemic_iqr,cirrhosis_liver_median,cirrhosis_liver_iqr,other_surgical_median,other_surgical_iqr,kruskal
hours_sleep_ecg_nn_all,6.2,"[4.4, 7.5]",6.0,"[3.2, 7.5]",6.7,"[1.3, 8.3]",7.4,"[5.0, 7.6]",5.9,"[4.4, 7.3]",7.3,"[4.7, 9.5]",6.1,"[4.9, 7.5]",5.5,"[3.6, 9.0]",6.6,"[4.6, 8.7]",6.9,"[6.0, 7.9]",7.8,"[5.5, 10.0]",6.4,"[6.4, 10.7]",7.0,"[5.0, 7.5]",4.8,"[3.9, 6.2]",6.8,"[4.9, 9.2]","7.7, 0.904"
hours_sleep_ecg_nn_agreement_relaxed,3.5,"[2.1, 4.1]",3.7,"[1.8, 4.7]",3.2,"[1.1, 5.0]",3.7,"[2.7, 4.1]",3.3,"[2.7, 4.5]",4.2,"[3.2, 6.4]",2.7,"[1.5, 3.8]",2.6,"[1.5, 4.3]",3.7,"[2.1, 5.2]",3.5,"[2.7, 4.4]",4.9,"[3.0, 6.4]",3.4,"[2.5, 3.9]",3.2,"[2.8, 3.7]",2.7,"[2.6, 3.3]",4.1,"[2.8, 5.7]","12.5, 0.565"
hours_discordant_ecg_nn_agreement_relaxed,2.8,"[1.0, 3.8]",1.7,"[0.6, 3.8]",2.0,"[0.3, 3.8]",2.3,"[0.6, 4.7]",2.3,"[1.3, 3.1]",3.2,"[0.5, 4.1]",3.1,"[2.3, 4.5]",2.3,"[0.8, 5.3]",2.8,"[0.6, 3.8]",3.6,"[1.5, 5.3]",2.6,"[1.3, 3.5]",4.8,"[3.4, 8.3]",3.7,"[2.6, 4.2]",1.7,"[1.2, 3.5]",2.3,"[1.1, 3.7]","8.0, 0.888"
discordant_proportion_ecg_nn_agreement_relaxed,0.38,"[0.17, 0.5]",0.31,"[0.14, 0.45]",0.43,"[0.08, 0.61]",0.54,"[0.17, 0.62]",0.36,"[0.2, 0.46]",0.32,"[0.12, 0.42]",0.49,"[0.28, 0.59]",0.51,"[0.25, 0.59]",0.35,"[0.12, 0.54]",0.53,"[0.41, 0.63]",0.36,"[0.35, 0.45]",0.75,"[0.53, 0.75]",0.5,"[0.45, 0.55]",0.54,"[0.31, 0.54]",0.34,"[0.18, 0.48]","16.6, 0.278"
perc_S_ecg_nn_all,0.42,"[0.22, 0.53]",0.37,"[0.25, 0.49]",0.41,"[0.07, 0.52]",0.45,"[0.25, 0.53]",0.32,"[0.27, 0.51]",0.42,"[0.39, 0.69]",0.37,"[0.25, 0.53]",0.37,"[0.25, 0.54]",0.43,"[0.32, 0.64]",0.42,"[0.38, 0.53]",0.49,"[0.32, 0.57]",0.62,"[0.54, 0.71]",0.39,"[0.34, 0.46]",0.32,"[0.26, 0.32]",0.5,"[0.3, 0.66]","10.8, 0.705"
perc_R_ecg_nn_all,0.04,"[0.0, 0.13]",0.04,"[0.0, 0.2]",0.04,"[0.01, 0.08]",0.01,"[0.0, 0.13]",0.0,"[0.0, 0.06]",0.34,"[0.04, 0.7]",0.07,"[0.0, 0.26]",0.1,"[0.0, 0.43]",0.06,"[0.0, 0.22]",0.19,"[0.08, 0.25]",0.05,"[0.02, 0.09]",0.04,"[0.02, 0.31]",0.11,"[0.03, 0.23]",0.34,"[0.3, 0.63]",0.12,"[0.02, 0.34]","18.4, 0.191"
perc_N1_ecg_nn_all,0.33,"[0.16, 0.82]",0.28,"[0.07, 0.77]",0.26,"[0.04, 0.36]",0.33,"[0.17, 0.82]",0.31,"[0.12, 0.78]",0.36,"[0.18, 0.6]",0.31,"[0.24, 0.81]",0.18,"[0.04, 0.31]",0.22,"[0.08, 0.46]",0.15,"[0.08, 0.28]",0.09,"[0.04, 0.15]",0.09,"[0.06, 0.19]",0.17,"[0.13, 0.25]",0.16,"[0.15, 0.19]",0.17,"[0.06, 0.41]","16.6, 0.279"
perc_N2_ecg_nn_all,0.01,"[0.0, 0.06]",0.05,"[0.0, 0.13]",0.01,"[0.0, 0.04]",0.04,"[0.02, 0.07]",0.03,"[0.01, 0.13]",0.03,"[0.0, 0.1]",0.06,"[0.01, 0.09]",0.02,"[0.0, 0.08]",0.02,"[0.0, 0.08]",0.03,"[0.01, 0.12]",0.03,"[0.01, 0.14]",0.01,"[0.01, 0.02]",0.05,"[0.02, 0.06]",0.14,"[0.01, 0.33]",0.11,"[0.01, 0.21]","15.7, 0.33"
perc_N3_ecg_nn_all,0.41,"[0.06, 0.6]",0.3,"[0.05, 0.51]",0.59,"[0.32, 0.89]",0.17,"[0.05, 0.48]",0.37,"[0.06, 0.72]",0.06,"[0.01, 0.18]",0.17,"[0.06, 0.41]",0.23,"[0.07, 0.58]",0.44,"[0.1, 0.59]",0.57,"[0.42, 0.62]",0.86,"[0.64, 0.91]",0.91,"[0.51, 0.91]",0.6,"[0.42, 0.68]",0.09,"[0.08, 0.12]",0.24,"[0.05, 0.55]","21.9, 0.081"
perc_N2N3_ecg_nn_all,0.46,"[0.15, 0.72]",0.3,"[0.07, 0.75]",0.63,"[0.36, 0.91]",0.18,"[0.08, 0.7]",0.54,"[0.15, 0.84]",0.1,"[0.05, 0.39]",0.3,"[0.15, 0.52]",0.31,"[0.18, 0.6]",0.5,"[0.11, 0.73]",0.62,"[0.49, 0.7]",0.87,"[0.77, 0.94]",0.91,"[0.52, 0.92]",0.7,"[0.54, 0.74]",0.43,"[0.22, 0.5]",0.45,"[0.18, 0.77]","18.4, 0.189"


In [33]:
%matplotlib widget
import matplotlib.pyplot as plt
import seaborn as sns

In [34]:
def get_variable_values(variable):

    list_median = []
    list_iqr = []
    list_n = []
    
    for diagnosis in diagnoses.keys():
        tmp_median = float(df_results.loc[variable, diagnosis + '_median'])
        tmp_iqr = df_results.loc[variable, diagnosis + '_iqr']
        tmp_q25 = float(tmp_iqr.split(',')[0].replace('[', ''))
        tmp_q75 = float(tmp_iqr.split(',')[1].replace(']', ''))

        list_median.append(tmp_median)
        list_iqr.append(np.array([tmp_median - tmp_q25, tmp_q75 - tmp_median]))
        list_n.append(df_results.loc['N', diagnosis + '_median'] )
        
    list_iqr = np.array(list_iqr).transpose()
    
    return list_median, list_iqr, list_n

In [35]:
len(diagnoses.keys())

15

In [36]:
y_labels = ['Total Sleep (h)', 'Concordant S. (h)', 'Discordant S. (h)', 'Discordant S. (%)', 'Sleep (%)', 'Stage R (%)', 'Stage N1 (%)', 'Stage N2 (%)', 'Stage N3 (%)', 'Stage N2+N3 (%)', 'SFI', 'Wake Trans./h']
# y_labels_short = ['TS', 'CS', 'DS', 'DS%', 'S', 'R', 'N1', 'N2', 'N3', 'N2+N3', 'SFI', 'WT']
y_labels_short = ['Total\nSleep (h)', 'Concor.\nSleep(h)', 'Discor.\nSleep (h)', 'Discor.\nSleep (%)', 'Sleep(%)', 'R (%)', 'N1 (%)', 'N2 (%)', 'N3 (%)', 'N2+N3\n(%)', 'SFI', 'Wake\nTrans./h']


In [37]:
df_results.index[:-1]

vars_to_plot = ['hours_sleep_ecg_nn_all', 'hours_sleep_ecg_nn_agreement_relaxed',
       'hours_discordant_ecg_nn_agreement_relaxed',
       'discordant_proportion_ecg_nn_agreement_relaxed', 'perc_S_ecg_nn_all',
       'perc_R_ecg_nn_all', 'perc_N1_ecg_nn_all', 'perc_N2N3_ecg_nn_all', 'sfi_ecg_nn_all',
       'arousali_ecg_nn_all']

y_labels = ['Total Sleep (h)', 'Concordant S. (h)', 'Discordant S. (h)', 'Discordant S. (%)', 'Sleep (%)', 'Stage R (%)', 'Stage N1 (%)', 'Stage N2+N3 (%)', 'SFI', 'Wake Trans./h']
# y_labels_short = ['TS', 'CS', 'DS', 'DS%', 'S', 'R', 'N1', 'N2', 'N3', 'N2+N3', 'SFI', 'WT']
y_labels_short = ['Total\nSleep (h)', 'Concor.\nSleep(h)', 'Discor.\nSleep (h)', 'Discor.\nSleep (%)', 'Sleep(%)', 'R (%)', 'N1 (%)', 'N2+N3\n(%)', 'SFI', 'Wake\nTrans./h']


In [38]:
list_median_DS, list_iqr_DS, list_n = get_variable_values('discordant_proportion_ecg_nn_agreement_relaxed')
argsort_discordant = np.argsort(list_median_DS)

In [39]:
diagnoses.keys()

dict_keys(['sepsis', 'shock', 'heart_failure', 'encephalopathy', 'anemia', 'hemorrhage', 'pneumonia', 'respiratory_failure', 'aki', 'pneumothorax', 'copd', 'other_medical', 'gi_hernia_ischemic', 'cirrhosis_liver', 'other_surgical'])

In [40]:
diagnoses_labels = {
    'sepsis': 'Sepsis',
    'shock': 'Shock',
    'heart_failure': 'Heart failure',
    'encephalopathy': 'Encephalopathy',
    'anemia': 'Anemia', 
    'hemorrhage': 'Hemorrhage', 
    'pneumonia': 'Pneumonia',
    'respiratory_failure': 'Respiratory failure',
    'aki': 'AKI',
    'pneumothorax': 'Pneumothorax',
    'copd': 'COPD',
    'other_medical': 'Other - medical',
    'gi_hernia_ischemic': 'GI, hernia,\nischemic colitis',
    'cirrhosis_liver': 'Liver transplant,\ncirrhosis',
    'other_surgical': 'Other - surgical'
}

diagnoses_colors = {
    'sepsis': 'green',
    'shock': 'green',
    'heart_failure': 'red',
    'encephalopathy': 'orange',
    'anemia': 'red', 
    'hemorrhage': 'red', 
    'pneumonia': 'blue',
    'respiratory_failure': 'blue',
    'aki': 'brown',
    'pneumothorax': 'blue',
    'copd': 'blue',
    'other_medical': 'gray',
    'gi_hernia_ischemic': 'brown',
    'cirrhosis_liver': 'brown',
    'other_surgical': 'gray',
}

todo:
    sort by DS%
    
    format xtick labels
    improve y labels
    improve order of variables?

In [41]:
fig, ax = plt.subplots(len(vars_to_plot) // 2, 2, figsize = (7.6, 5), sharex=True)
ax = ax.flatten()

for iv, variable in enumerate(vars_to_plot):

    list_median, list_iqr, list_n = get_variable_values(variable)
    list_median = np.array(list_median)[argsort_discordant]
    list_iqr = [np.array(list_iqr[0])[argsort_discordant], np.array(list_iqr[1])[argsort_discordant]]
    list_n = np.array(list_n)[argsort_discordant]
    
    diagnoses_sorted = np.array(list(diagnoses.keys()))[argsort_discordant]
    
    sns.barplot(x=np.arange(len(diagnoses.keys())), y=list_median, 
                linewidth=1, yerr = list_iqr, errwidth=0.01,
#                 facecolor=(1, 1, 1, 0),
                palette = [diagnoses_colors[x] for x in diagnoses_sorted],
                errcolor=".2", edgecolor=".2", ax = ax[iv])

    ax[iv].set_ylabel(y_labels_short[iv])
    
    if variable == 'discordant_proportion_ecg_nn_agreement_relaxed':
        ax[iv].set_ylim([0, 1])
        ax[iv].set_yticks([0, 0.25, 0.5, 0.75])
    elif variable == 'arousali_ecg_nn_all':
        ax[iv].set_ylim([0, 15])
        ax[iv].set_yticks([0, 5, 10])
        
    
ax[-2].set_xticklabels([diagnoses_labels[x] + f' ({int(n)})' for x, n in zip(diagnoses_sorted, list_n)], rotation=90)
ax[-1].set_xticklabels([diagnoses_labels[x] + f' ({int(n)})' for x, n in zip(diagnoses_sorted, list_n)], rotation=90)

plt.subplots_adjust(hspace=0, left=0.07, right=0.995, bottom=0.25, top=0.995)

plt.savefig('sleep_statistics_diagnoses_groups.svg')
plt.savefig('sleep_statistics_diagnoses_groups.eps')
plt.savefig('sleep_statistics_diagnoses_groups.tiff', dpi=400)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [47]:
plt.figure()

sns.barplot(x=[0, 1], y=[2, 3], 

                 linewidth=2.5, yerr = [[1, 0.2], [0.1, 0.2]])

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

<AxesSubplot:>